# Using the CLI tool and iterating on CI checks (Edition 2)

The purpose of this workbook is to get familiar with the workflows around creating and submitting Terminal-Bench **Edition 2** tasks. We will go through how to:

1. Start a task using the CLI tool (or platform skeleton)
2. Create an initial task submission to get automated feedback
3. Read and iterate on **Harbor / platform CI** feedback

Your goal is to learn how to refine a task **before** human review. This training uses a deliberately incomplete hello-world task in `default-template.zip`.

**Edition 2 changes to know up front:**
- **No legacy canary strings** — do not add `# terminal-bench-canary …` to any file; `check_canary` rejects canary text in `instruction.md`.
- **Digest-pinned Docker images** — every `FROM` must include `@sha256:<digest>` (`check_pinned_images`).
- **Reward block ends `test.sh`** — write `/logs/verifier/reward.txt`, then stop; no trailing `exit` after the reward `if`/`else` (`check_test_sh`).
- **`allow_internet = false`** — install pytest and verifier tooling in `environment/Dockerfile`, not in `tests/test.sh` at runtime.


## Step 1: Start a task using the CLI tool

Starting a task locally gives you:

1. A task directory for your work
2. Placeholder files for Edition 2 layout: `task.toml`, `instruction.md`, `environment/Dockerfile`, `solution/solve.sh`, `tests/test.sh`, `tests/test_outputs.py`
3. A `tests/` directory for pytest verifiers

Download the current skeleton from the platform **Resources** tab, or unzip `default-template.zip` from this training bundle.


### Preparing the repo (required)

If you haven't already, clone the task repository your cohort uses (for example `snorkel-tb-tasks` or your Terminus Edition 2 fork).

Create a training branch:

```bash
git checkout -b training/<your-github-name>-training
```


### Install uv (required)

Install **uv** to run the CLI and offline pytest helpers:

https://docs.astral.sh/uv/getting-started/installation/

For local CI iteration you also need **Harbor**:

```bash
harbor tasks check <task-folder> -m openai/@openai/gpt-5.5
```


### Creating the task

In your terminal, `cd` into the repository, then create a task directory.

**Option A — platform / stb CLI**

```bash
uv run stb tasks create
```

**Option B — copy the training skeleton**

```bash
unzip default-template.zip -d tasks/training-<your-github-username>
```

Follow the console prompts (Option A) or edit the copied files (Option B).


## Step 2: Creating your submission

Once the task directory exists, you would normally author a real task. For this training, use the provided hello-world skeleton only.

See the platform docs for **Creating Docker Environment**, **Writing Tests**, and **Platform Submission Guide**, or your cohort's Edition 2 authoring guide.


### Copying over the sample task

For this training, copy the incomplete skeleton into the directory you created:

```bash
unzip -o default-template.zip -d tasks/training-<your-github-username>
```

The sample task is intentionally easy and **intentionally fails two blocking CI checks** until you fix them:

1. `check_pinned_images` — Dockerfile `FROM` is missing `@sha256:<digest>`
2. `check_test_sh` — `tests/test.sh` has a trailing `exit` after the reward block

Read the `TRAINING NOTE` comments in those files.


### Committing our changes

Commit **only** your training task directory:

```bash
git add tasks/training-<your-github-username>
git commit -m "initial training task submission"
git push --set-upstream origin training/<your-github-username>-training
```

Open a PR (GitHub workflow) **or** upload a zip to the Terminus Edition 2 platform (platform workflow) so automated checks run.


### Iterating on CI feedback — Example 1: `check_pinned_images`

Run locally:

```bash
harbor tasks check tasks/training-<your-github-username> -m openai/@openai/gpt-5.5
```

When `check_pinned_images` fails, open `environment/Dockerfile` and pin **every** `FROM` stage:

```dockerfile
FROM python:3.13-slim@sha256:<digest>
```

Look up the digest from Docker Hub / your registry, or from a pinned image in the task gallery. Tag-only `FROM python:3.13-slim` is blocked.

Also verify:
- Final runtime base should be a **§2 canonical Terminal-Bench image** (see `docs/edition2/DOCKERFILE_BEST_PRACTICES.md`) or include `# BASE IMAGE JUSTIFICATION:` / environment README note
- `environment/` stays ≤ **100 MiB** total and ≤ **50 MiB** per file
- `tmux` and `asciinema` are installed (Harbor agent runtime requirement)


### Iterating on CI feedback — Example 2: `check_test_sh`

Edition 2 `tests/test.sh` must **always** write `/logs/verifier/reward.txt` and end at the reward block. Harbor scores `reward.txt`, not the shell exit code.

**Remove** the trailing `exit "$RC"` (or any `exit` after the reward `if`/`else`).

Correct tail:

```bash
if [ $? -eq 0 ]; then
  echo 1 > /logs/verifier/reward.txt
else
  echo 0 > /logs/verifier/reward.txt
fi
```

**Do not** flag a missing trailing exit during review — it is correct Edition 2 shape.

Verifier dependencies (pytest, plugins, `uv` if used) belong in `environment/Dockerfile` because `[environment] allow_internet = false`.


## Step 3: Finish the rest of CI

Review remaining failures one at a time. Common **blocking** checks:

| Check | Fix |
|-------|-----|
| `check_pinned_images` | Add `@sha256:<digest>` to every `FROM` |
| `check_sanctioned_base_images` | §2 canonical TB base or justified non-canonical final stage |
| `check_build_context_size` | Trim `environment/` under 100 MiB / 50 MiB per file |
| `check_test_sh` | Reward block is script end; no trailing `exit` |
| `check_canary` | Remove legacy canary strings from `instruction.md` (do **not** add them) |
| `pinned_dependencies` | Pin apt/pip/npm versions in Dockerfile |
| `check_task_absolute_path` | Use absolute paths like `/app/...` in `instruction.md` |
| `ruff` | `ruff check --fix <task-folder>` |

LLMaJ checks (`behavior_in_tests`, `anti_cheating_measures`, …) run via the same `harbor tasks check` command — investigate warnings even when not blocking.

Re-run until blocking CI passes, then run **oracle** (must PASS) and **NOP** (must score 0.0):

```bash
harbor run -a oracle -p tasks/training-<your-github-username>
```

When all blocking CI checks pass and oracle/NOP look good, you have finished this training module.


### Solution

If you get stuck, compare your fixes to:

- Pinned `FROM` in `environment/Dockerfile`
- Reward-only ending in `tests/test.sh` (no trailing `exit`)
- `[environment] allow_internet = false` with pytest pre-installed in the Dockerfile

Your cohort may also provide a walkthrough video for this module.
